# ML Network Intrusion Detector
**IBM SkillsBuild Cyber Security BootCamp 2026**  
**Author:** Vanshika Maru | **Email:** vanshikamaru0205@gmail.com

This educational project classifies labeled network-flow records as Normal or Attack using a Random Forest. It is a prototype, not a live network monitor. Download a labeled CIC-IDS2017 CSV and configure `DATA_PATH` before running.

## 1. Imports and configuration
Install dependencies with `pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
import json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATA_PATH = Path('data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load dataset
Download a labeled CSV from the [official CIC-IDS2017 page](https://www.unb.ca/cic/datasets/ids-2017.html). Update `DATA_PATH` if your filename differs.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}. Download a labeled CIC-IDS2017 CSV into data/ and update DATA_PATH.')
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.astype(str).str.strip()
print('Dataset shape:', df.shape)
display(df.head())

## 3. Clean data and prepare labels
`BENIGN` is mapped to Normal; every other label is mapped to Attack.

In [ ]:
label_col = next((c for c in df.columns if c.strip().lower() == 'label'), None)
if label_col is None: raise ValueError('No Label column found in the CSV.')
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col]).drop_duplicates()
y_raw = df[label_col].astype(str).str.strip()
y = np.where(y_raw.str.upper().eq('BENIGN'), 'Normal', 'Attack')
X = df.drop(columns=[label_col]).copy()
X = X.loc[:, ~X.columns.duplicated()].dropna(axis=1, how='all')
print(pd.Series(y).value_counts())
pd.Series(y).value_counts().plot(kind='bar', title='Class Distribution')
plt.xlabel('Class'); plt.ylabel('Records'); plt.tight_layout(); plt.show()

## 4. Split and preprocess
Missing numeric values are imputed with the median. Categorical values are imputed with the most frequent value and one-hot encoded.

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
numeric = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])
prep = ColumnTransformer([('num', numeric, num_cols), ('cat', categorical, cat_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print('Training:', len(X_train), 'Testing:', len(X_test))

## 5. Train model

In [ ]:
model = Pipeline([('preprocessor', prep), ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
model.fit(X_train, y_train)
print('Training complete.')

## 6. Evaluate performance
Use these actual test-set results in the project report after running the notebook. Do not invent metrics.

In [ ]:
pred = model.predict(X_test)
metrics = {'accuracy': float(accuracy_score(y_test, pred)), 'macro_precision': float(precision_score(y_test, pred, average='macro', zero_division=0)), 'macro_recall': float(recall_score(y_test, pred, average='macro', zero_division=0)), 'macro_f1': float(f1_score(y_test, pred, average='macro', zero_division=0))}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, pred, zero_division=0))
with open(OUTPUT_DIR / 'evaluation_metrics.json', 'w', encoding='utf-8') as f: json.dump(metrics, f, indent=2)
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap='Blues')
plt.title('Confusion Matrix — Test Set'); plt.tight_layout(); plt.show()

## 7. Save outputs and conclusion
This is a learning prototype. Results depend on the chosen dataset and split; unseen attacks and changing network patterns can reduce performance. It does not inspect live packets or block traffic.

In [ ]:
joblib.dump(model, OUTPUT_DIR / 'network_intrusion_random_forest.joblib')
print('Saved model and evaluation_metrics.json to outputs/.')